In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
!pip install biopython

In [ ]:
#!apt-get install pymol

In [ ]:
import os
import numpy as np
import pandas as pd
from Bio import PDB

# Method 1

In [ ]:
def merge_pdb_files(input_dir, output_file):
    parser = PDB.PDBParser(QUIET=True)
    io = PDB.PDBIO()

    pdb_files = sorted([f for f in os.listdir(input_dir) if f.endswith(".pdb")])
    if not pdb_files:
        print("No PDB files found in the directory.")
        return

    base_structure = parser.get_structure("base", os.path.join(input_dir, '7e8c.pdb'))
    base_model = base_structure[0]
    structures = [parser.get_structure(pdb_file, os.path.join(input_dir, pdb_file)) for pdb_file in pdb_files[1:]]

    for structure in structures:
        for chain in structure[0]:
            if chain.get_id() not in base_model:
                base_model.add(chain)

    io.set_structure(base_structure)
    io.save(output_file)
    print(f"Merged {len(pdb_files)} PDB files into {output_file}")

In [ ]:
# Set input directory and output file
input_directory = "/content/drive/MyDrive/@antigen_epitope/abdb_combine/test"
output_pdb_file = "/content/drive/MyDrive/@antigen_epitope/abdb_combine/nAbs_combined.pdb"

merge_pdb_files(input_directory, output_pdb_file)

# Method 2

In [ ]:
parser = PDB.PDBParser(QUIET=True)
structure_big = parser.get_structure("big_protein", "/content/drive/MyDrive/@antigen_epitope/abdb_combine/nAbs/7e8c.pdb")
structure_small = parser.get_structure("small_protein", "/content/drive/MyDrive/@antigen_epitope/abdb_combine/nAbs_combined.pdb")

def get_unoverlapped_residues(big_structure, small_structure, cutoff=2.0):
    big_atoms = [atom for residue in big_structure.get_residues() for atom in residue]
    small_residues = []

    for residue in small_structure.get_residues():
        overlap = False
        for atom in residue:
            for big_atom in big_atoms:
                if atom - big_atom < cutoff:
                    overlap = True
                    break
            if overlap:
                break
        if not overlap:
            small_residues.append(residue)

    return small_residues

unoverlapped_residues = get_unoverlapped_residues(structure_big, structure_small)

class UnoverlappedSelect(PDB.Select):
    def __init__(self, residues_to_keep):
        self.residues_to_keep = residues_to_keep

    def accept_residue(self, residue):
        return residue in self.residues_to_keep

io = PDB.PDBIO()
io.set_structure(structure_small)
io.save("/content/unoverlapped_small.pdb", select=UnoverlappedSelect(unoverlapped_residues))

print("Saved unoverlapped regions to /content/unoverlapped_small.pdb")

In [ ]:
!cp /content/unoverlapped_small.pdb /content/drive/MyDrive/@antigen_epitope/abdb_combine/